# A chain of FitzHugh-Nagumo cells: waves

20 diffusively coupled FitzHugh-Nagumo cells written with XPP's arrays. Cell 1 is
driven by `i1` and fires repeatedly; each spike travels down the chain. `%[1..20] ... %`
repeats the lines with [j] = 1, 2, ..., 20; the ends are no-flux.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.1 and its AUTO through pyauto-xpp.*

In [1]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [2]:
mb = (ModelBuilder("fhn_chain")
      .begin_block("1..20")
      .ode("v[j]", "v[j]-v[j]^3/3-w[j]+i1*heav(1.5-[j])+d*(v[j+1]-2*v[j]+v[j-1])")
      .end_block()
      .ode("w[1..20]", "eps*(v[j]+a-b*w[j])")
      .fixed("v0", "v1")
      .fixed("v21", "v20")
      .parameter(i1=1, d=1, eps=0.08, a=0.7, b=0.8)
      .initial({"v[1..20]": -1.2})
      .initial({"w[1..20]": -0.62})
      .option(total=200, dt=0.05))
print(mb.render())

%[1..20]
v[j]'=v[j]-v[j]^3/3-w[j]+i1*heav(1.5-[j])+d*(v[j+1]-2*v[j]+v[j-1])
%
w[1..20]'=eps*(v[j]+a-b*w[j])
v0=v1
v21=v20
par i1=1,d=1,eps=0.08,a=0.7,b=0.8
init v[1..20]=-1.2
init w[1..20]=-0.62
@ total=200,dt=0.05
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [3]:
ex = Explorer(mb, kind="array", cells=("v", ["v1", "v10", "v20"]), sliders={"i1": (0, 1.5), "d": (0, 3)},
              title="FitzHugh-Nagumo chain")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Wave speed**: the time a spike takes from cell 1 to cell 20 against the coupling `d`.

In [4]:
import numpy as np
D = np.linspace(0.3, 3, 10)
speed = []
for d in D:
    tr = ex.m.run(keep=False, d=d, total=200)
    t1 = tr.t[np.argmax(tr["v1"] > 1)]; t20 = tr.t[np.argmax(tr["v20"] > 1)]
    speed.append(19 / (t20 - t1) if t20 > t1 else np.nan)
show_figure(go.Figure(go.Scatter(x=D, y=speed, mode="lines+markers")), xaxis_title="d",
            yaxis_title="cells per time unit", height=300)

## Things to try

- Speed should grow like the square root of `d` (diffusion).